# 08 · Relationship layer (twin network graph)

Builds the **views-first relationship layer** that turns the five digital twins into one typed
graph: **`v_network_nodes`** and **`v_network_edges`** in `monitoring`. These power the
interactive **twin-network graph + node drilldown** in the Twin Explorer app (and, later, Genie
and the agent).

**Nodes:** MATERIAL, SUPPLIER, PLANT, WORK_CENTRE, CHANNEL, CUSTOMER, REGION — each colored by its
twin `status` (GREEN/AMBER/RED) and carrying key metrics.
**Edges:** SUPPLIES, COMPONENT_OF (BOM), PRODUCED_AT, STOCKED_AT, ROUTED_THROUGH, SHIPS_TO, SOLD_TO.

Views over `gold` + the twin tables (live, no copy). Materialize later only if it gets slow.

_Data limits: distribution edges resolve to ship-to **region** (no outlet/DC network);
`ROUTED_THROUGH` uses the primary work centre only (no full routing master); `SOLD_TO` uses the
top customer per series; BOM edges from `gold.billofmaterials`._

## Config

In [ ]:
dbutils.widgets.text("source_catalog", "mickmickstableaws_catalog", "Source catalog")
dbutils.widgets.text("source_schema", "gold", "Source schema")
dbutils.widgets.text("target_catalog", "mickmickstableaws_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")

SRC = f"{dbutils.widgets.get('source_catalog').strip()}.{dbutils.widgets.get('source_schema').strip()}"
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
print(f"Source: {SRC}\nOutputs (views): {FQ_MON}.v_network_nodes , {FQ_MON}.v_network_edges")

## 1. `v_network_nodes`
Typed nodes with health status (from twin risk flags) and a `metric_json` for tooltips / drilldown.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {FQ_MON}.v_network_nodes (
  node_id     COMMENT 'Unique node id, formatted TYPE:key',
  node_type   COMMENT 'MATERIAL | SUPPLIER | PLANT | WORK_CENTRE | CHANNEL | CUSTOMER | REGION',
  label       COMMENT 'Human-readable node label',
  plant       COMMENT 'Plant the node belongs to (nullable)',
  node_group  COMMENT 'Grouping attribute (material group, supplier country, etc.)',
  status      COMMENT 'Health derived from twin risk flags: GREEN | AMBER | RED',
  metric_json COMMENT 'JSON of key twin metrics for tooltip / drilldown'
) COMMENT 'Typed nodes of the supply-chain twin graph.' AS
WITH inv AS (
  SELECT material, SUM(on_hand_qty) AS on_hand, MIN(days_of_cover_demand) AS min_doc,
         MAX(CAST(is_stockout AS INT)) AS any_stockout, MAX(CAST(is_low_cover AS INT)) AS any_low,
         MAX(CAST(is_overstock AS INT)) AS any_over
  FROM {FQ_MON}.inventory_twin_current GROUP BY material
),
dmd AS (
  SELECT material, SUM(horizon_total_forecast_qty) AS horizon_fc, AVG(forecast_wmape) AS wmape
  FROM {FQ_MON}.demand_twin_current GROUP BY material
),
prd AS (
  SELECT material, AVG(yield_pct) AS yield_pct, SUM(open_production_qty) AS open_prod
  FROM {FQ_MON}.production_twin_current GROUP BY material
)
SELECT concat('MATERIAL:', m.Materialnumber_Matnr) AS node_id,
  CASE WHEN m.Materialtype_Mtart IN ('FERT','HALB','HAWA') OR fin.mat IS NOT NULL
       THEN 'FINISHED_GOOD' ELSE 'RAW_MATERIAL' END AS node_type,
  COALESCE(m.Materialtext_Maktx, m.Materialnumber_Matnr) AS label,
  CAST(NULL AS STRING) AS plant, m.Materialgroup_Matkl AS node_group,
  CASE WHEN COALESCE(inv.any_stockout,0)=1 OR COALESCE(inv.any_low,0)=1 THEN 'RED'
       WHEN COALESCE(inv.any_over,0)=1 THEN 'AMBER' ELSE 'GREEN' END AS status,
  to_json(named_struct('material_type', m.Materialtype_Mtart, 'on_hand', inv.on_hand,
    'min_days_of_cover', inv.min_doc, 'horizon_forecast', dmd.horizon_fc,
    'forecast_wmape', dmd.wmape, 'yield_pct', prd.yield_pct, 'open_production_qty', prd.open_prod)) AS metric_json
FROM {SRC}.materialsmd m
LEFT JOIN inv ON m.Materialnumber_Matnr = inv.material
LEFT JOIN dmd ON m.Materialnumber_Matnr = dmd.material
LEFT JOIN prd ON m.Materialnumber_Matnr = prd.material
LEFT JOIN (SELECT DISTINCT FinishedMaterial_MATNR AS mat FROM {SRC}.billofmaterials WHERE IsActive = true) fin ON m.Materialnumber_Matnr = fin.mat
WHERE m.Materialnumber_Matnr IS NOT NULL
UNION ALL
SELECT concat('SUPPLIER:', supplier), 'SUPPLIER', MAX(supplier_name), CAST(NULL AS STRING), MAX(supplier_country),
  CASE WHEN MAX(CAST(supplier_risk_flag AS INT))=1 THEN 'RED' ELSE 'GREEN' END,
  to_json(named_struct('avg_otif', AVG(otif_rate), 'avg_lead_time_days', AVG(avg_lead_time_days), 'open_po_qty', SUM(open_po_qty)))
FROM {FQ_MON}.procurement_twin_current WHERE supplier IS NOT NULL GROUP BY supplier
UNION ALL
SELECT concat('PLANT:', plant), 'PLANT', concat('Plant ', plant), plant, CAST(NULL AS STRING), 'GREEN',
  to_json(named_struct('type', 'plant'))
FROM (SELECT DISTINCT plant FROM {FQ_MON}.inventory_twin_current
      UNION SELECT DISTINCT plant FROM {FQ_MON}.production_twin_current
      UNION SELECT DISTINCT plant FROM {FQ_MON}.distribution_twin_current) p
WHERE plant IS NOT NULL
UNION ALL
SELECT concat('WORKCENTRE:', Plant_WERKS, ':', WorkCentreCode_ARBPL), 'WORK_CENTRE', WorkCentreCode_ARBPL, Plant_WERKS, CAST(NULL AS STRING),
  CASE WHEN MAX(CAST(IsBottleneckFlag AS INT))=1 THEN 'RED' ELSE 'GREEN' END,
  to_json(named_struct('capacity_minutes', MAX(EffectiveDailyCapacityMinutes)))
FROM {SRC}.workcentres WHERE WorkCentreCode_ARBPL IS NOT NULL GROUP BY Plant_WERKS, WorkCentreCode_ARBPL
UNION ALL
SELECT concat('CHANNEL:', channel), 'CHANNEL', concat('Channel ', channel), CAST(NULL AS STRING), CAST(NULL AS STRING), 'GREEN',
  to_json(named_struct('type', 'channel'))
FROM (SELECT DISTINCT channel FROM {FQ_MON}.demand_twin_current WHERE channel IS NOT NULL) c
UNION ALL
SELECT concat('CUSTOMER:', top_customer_id), 'CUSTOMER', MAX(top_customer_name), CAST(NULL AS STRING), MAX(top_customer_region), 'GREEN',
  to_json(named_struct('type', 'customer'))
FROM {FQ_MON}.demand_twin_current WHERE top_customer_id IS NOT NULL GROUP BY top_customer_id
UNION ALL
SELECT concat('REGION:', top_ship_to_region), 'REGION', concat('Region ', top_ship_to_region), CAST(NULL AS STRING), CAST(NULL AS STRING), 'GREEN',
  to_json(named_struct('type', 'region'))
FROM (SELECT DISTINCT top_ship_to_region FROM {FQ_MON}.distribution_twin_current WHERE top_ship_to_region IS NOT NULL) r
""")
print("v_network_nodes created")
display(spark.sql(f"SELECT node_type, COUNT(*) nodes, SUM(CASE WHEN status='RED' THEN 1 ELSE 0 END) red FROM {FQ_MON}.v_network_nodes GROUP BY node_type ORDER BY node_type"))

## 2. `v_network_edges`
Typed edges with a `weight` (volume/quantity) and `attrs_json` (lead time, OTIF, share, ...).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {FQ_MON}.v_network_edges (
  source_id  COMMENT 'Source node id',
  target_id  COMMENT 'Target node id',
  edge_type  COMMENT 'SUPPLIES | COMPONENT_OF | PRODUCED_AT | STOCKED_AT | ROUTED_THROUGH | SHIPS_TO | SOLD_TO',
  weight     COMMENT 'Edge magnitude (volume / quantity / share)',
  attrs_json COMMENT 'JSON of edge attributes (lead time, OTIF, share, ...)'
) COMMENT 'Typed edges of the supply-chain twin graph.' AS
SELECT concat('SUPPLIER:', supplier) AS source_id, concat('MATERIAL:', material) AS target_id, 'SUPPLIES' AS edge_type,
  CAST(SUM(po_qty) AS DOUBLE) AS weight,
  to_json(named_struct('avg_otif', AVG(otif_rate), 'avg_lead_time_days', AVG(avg_lead_time_days))) AS attrs_json
FROM {FQ_MON}.procurement_twin_current WHERE supplier IS NOT NULL AND material IS NOT NULL GROUP BY supplier, material
UNION ALL
SELECT concat('MATERIAL:', ComponentMaterial_IDNRK), concat('MATERIAL:', FinishedMaterial_MATNR), 'COMPONENT_OF',
  CAST(SUM(ComponentQuantity_MENGE) AS DOUBLE), to_json(named_struct('bom_level', MAX(BOMLevel)))
FROM {SRC}.billofmaterials
WHERE IsActive = true AND ComponentMaterial_IDNRK IS NOT NULL AND FinishedMaterial_MATNR IS NOT NULL
GROUP BY ComponentMaterial_IDNRK, FinishedMaterial_MATNR
UNION ALL
SELECT concat('MATERIAL:', material), concat('PLANT:', plant), 'PRODUCED_AT',
  CAST(SUM(produced_qty) AS DOUBLE), to_json(named_struct('open_production_qty', SUM(open_production_qty)))
FROM {FQ_MON}.production_twin_current WHERE material IS NOT NULL AND plant IS NOT NULL GROUP BY material, plant
UNION ALL
SELECT concat('MATERIAL:', material), concat('PLANT:', plant), 'STOCKED_AT',
  CAST(SUM(on_hand_qty) AS DOUBLE), to_json(named_struct('min_days_of_cover', MIN(days_of_cover_demand)))
FROM {FQ_MON}.inventory_twin_current WHERE material IS NOT NULL AND plant IS NOT NULL GROUP BY material, plant
UNION ALL
SELECT concat('MATERIAL:', material), concat('WORKCENTRE:', plant, ':', primary_work_centre), 'ROUTED_THROUGH',
  CAST(SUM(produced_qty) AS DOUBLE), to_json(named_struct('is_bottleneck', MAX(CAST(is_bottleneck AS INT))))
FROM {FQ_MON}.production_twin_current WHERE material IS NOT NULL AND primary_work_centre IS NOT NULL GROUP BY material, plant, primary_work_centre
UNION ALL
SELECT concat('MATERIAL:', material), concat('REGION:', top_ship_to_region), 'SHIPS_TO',
  CAST(SUM(delivered_qty) AS DOUBLE), to_json(named_struct('avg_otif', AVG(otif_rate)))
FROM {FQ_MON}.distribution_twin_current WHERE material IS NOT NULL AND top_ship_to_region IS NOT NULL GROUP BY material, top_ship_to_region
UNION ALL
SELECT concat('MATERIAL:', material), concat('CUSTOMER:', top_customer_id), 'SOLD_TO',
  CAST(AVG(top_customer_share) AS DOUBLE), to_json(named_struct('top_customer_share', AVG(top_customer_share)))
FROM {FQ_MON}.demand_twin_current WHERE material IS NOT NULL AND top_customer_id IS NOT NULL GROUP BY material, top_customer_id
""")
print("v_network_edges created")
display(spark.sql(f"SELECT edge_type, COUNT(*) edges FROM {FQ_MON}.v_network_edges GROUP BY edge_type ORDER BY edge_type"))

## 3. Verify

In [ ]:
print("=== node counts by type ===")
display(spark.sql(f"SELECT node_type, COUNT(*) n FROM {FQ_MON}.v_network_nodes GROUP BY node_type ORDER BY node_type"))
print("=== edge counts by type ===")
display(spark.sql(f"SELECT edge_type, COUNT(*) n FROM {FQ_MON}.v_network_edges GROUP BY edge_type ORDER BY edge_type"))
print("=== dangling edges (endpoint not in nodes) ===")
display(spark.sql(f"""
WITH n AS (SELECT node_id FROM {FQ_MON}.v_network_nodes)
SELECT COUNT(*) AS dangling FROM {FQ_MON}.v_network_edges e
LEFT JOIN n s ON e.source_id = s.node_id LEFT JOIN n t ON e.target_id = t.node_id
WHERE s.node_id IS NULL OR t.node_id IS NULL
"""))
print("=== sample ego graph for one material ===")
display(spark.sql(f"""
WITH pick AS (SELECT node_id FROM {FQ_MON}.v_network_nodes WHERE node_type='MATERIAL' ORDER BY node_id LIMIT 1)
SELECT e.* FROM {FQ_MON}.v_network_edges e JOIN pick ON e.source_id = pick.node_id OR e.target_id = pick.node_id
"""))

## Next
Relationship layer live. Next: extend the **Twin Explorer app** with a `Network` tab
(`streamlit-agraph`) reading these views — a material-centric ego graph + domain networks, with
node-click drilldown into each node's twin state and operational rows.